In [0]:
%pip install faker

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
CATALOG_NAME = "landing_raw"
SCHEMA_NAME = "customer_raw"
table_name = "customers"
FULL_TABLE_PATH = f"{CATALOG_NAME}.{SCHEMA_NAME}.{table_name}"
NUM_RECORDS = 1000

In [0]:
from faker import Faker
import random
import uuid
from datetime import datetime, timedelta

fake = Faker()
Faker.seed(42)
random.seed(42)

def generate_customers(n: int) -> list[dict]:
    records = []
    tiers = ["Bronze", "Silver", "Gold", "Platinum"]

    for _ in range(n):
        signup_date = fake.date_between(start_date="-5y", end_date="today")
        last_login  = signup_date + timedelta(days=random.randint(0, 365))

        records.append({
            "customer_id"    : str(uuid.uuid4()),
            "first_name"     : fake.first_name(),
            "last_name"      : fake.last_name(),
            "email"          : fake.email(),
            "phone"          : fake.phone_number(),
            "city"           : fake.city(),
            "country"        : fake.country(),
            "age"            : random.randint(18, 75),
            "gender"         : random.choice(["Male", "Female", "Non-binary"]),
            "signup_date"    : str(signup_date),
            "last_login_date": str(last_login),
            "is_active"      : random.choice([True, False]),
            "customer_tier"  : random.choice(tiers),
            "total_spent_usd": round(random.uniform(0, 15000), 2),
        })
    return records

raw_data = generate_customers(NUM_RECORDS)
print(f"✅ Generated {len(raw_data)} raw customer records")
for i in range(10):
    print("Sample:", raw_data[i])

✅ Generated 1000 raw customer records
Sample: {'customer_id': 'b02b30b1-0279-4dff-92a5-69968a4ba3e0', 'first_name': 'Angel', 'last_name': 'Hill', 'email': 'donaldgarcia@example.net', 'phone': '+1-219-560-0133', 'city': 'Robinsonshire', 'country': 'Fiji', 'age': 25, 'gender': 'Male', 'signup_date': '2024-07-29', 'last_login_date': '2025-06-21', 'is_active': False, 'customer_tier': 'Silver', 'total_spent_usd': 3348.16}
Sample: {'customer_id': '6fc4253a-e2de-41ac-9269-01fb03bc4b22', 'first_name': 'Adam', 'last_name': 'Shaffer', 'email': 'jpeterson@example.org', 'phone': '001-851-316-1559x40781', 'city': 'Mariastad', 'country': 'Morocco', 'age': 61, 'gender': 'Non-binary', 'signup_date': '2025-06-04', 'last_login_date': '2025-07-26', 'is_active': True, 'customer_tier': 'Platinum', 'total_spent_usd': 476.74}
Sample: {'customer_id': 'e0dd6a1a-bcfa-4206-a205-ac0efbf85c97', 'first_name': 'Chad', 'last_name': 'Stanley', 'email': 'tracie31@example.com', 'phone': '575-425-5341x928', 'city': 'Gray

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import *

# Define schema explicitly
schema = StructType([
    StructField("customer_id",     StringType(),  False),
    StructField("first_name",      StringType(),  True),
    StructField("last_name",       StringType(),  True),
    StructField("email",           StringType(),  True),
    StructField("phone",           StringType(),  True),
    StructField("city",            StringType(),  True),
    StructField("country",         StringType(),  True),
    StructField("age",             IntegerType(), True),
    StructField("gender",          StringType(),  True),
    StructField("signup_date",     StringType(),  True),
    StructField("last_login_date", StringType(),  True),
    StructField("is_active",       BooleanType(), True),
    StructField("customer_tier",   StringType(),  True),
    StructField("total_spent_usd", DoubleType(),  True),
])

df_raw = spark.createDataFrame(raw_data, schema=schema)

# --- Transformations ---
df_transformed = (
    df_raw
    .withColumn("signup_date",      F.to_date("signup_date",     "yyyy-MM-dd"))
    .withColumn("last_login_date",  F.to_date("last_login_date", "yyyy-MM-dd"))
    .withColumn("full_name",        F.concat_ws(" ", F.col("first_name"), F.col("last_name")))
    .withColumn("email",            F.lower(F.col("email")))
    .withColumn("days_since_signup", F.datediff(F.current_date(), F.col("signup_date")))
    .withColumn("spend_segment",
        F.when(F.col("total_spent_usd") < 500,  "Low")
         .when(F.col("total_spent_usd") < 3000, "Medium")
         .when(F.col("total_spent_usd") < 8000, "High")
         .otherwise("VIP"))
    .withColumn("ingested_at", F.current_timestamp())
    .dropna(subset=["customer_id", "email"])
)

print(f"✅ Transformed: {df_transformed.count()} records")
df_transformed.printSchema()

✅ Transformed: 1000 records
root
 |-- customer_id: string (nullable = false)
 |-- first_name: string (nullable = true)
 |-- last_name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- phone: string (nullable = true)
 |-- city: string (nullable = true)
 |-- country: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- gender: string (nullable = true)
 |-- signup_date: date (nullable = true)
 |-- last_login_date: date (nullable = true)
 |-- is_active: boolean (nullable = true)
 |-- customer_tier: string (nullable = true)
 |-- total_spent_usd: double (nullable = true)
 |-- full_name: string (nullable = false)
 |-- days_since_signup: integer (nullable = true)
 |-- spend_segment: string (nullable = false)
 |-- ingested_at: timestamp (nullable = false)



In [0]:
display(df_raw)

In [0]:
display(df_transformed)

In [0]:
# Ensure the schema exists (catalog must already exist)
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG_NAME}.{SCHEMA_NAME}")

# Write as a managed Delta table
(
    df_transformed
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(FULL_TABLE_PATH)
)

print(f"✅ Data loaded into: {FULL_TABLE_PATH}")

✅ Data loaded into: landing_raw.customer_raw.customers


In [0]:
df_check = spark.read.table(FULL_TABLE_PATH)

print(f"Total rows : {df_check.count()}")
print(f"Total cols : {len(df_check.columns)}")
display(df_check.limit(10))

Total rows : 1000
Total cols : 18


customer_id,first_name,last_name,email,phone,city,country,age,gender,signup_date,last_login_date,is_active,customer_tier,total_spent_usd,full_name,days_since_signup,spend_segment,ingested_at
b02b30b1-0279-4dff-92a5-69968a4ba3e0,Angel,Hill,donaldgarcia@example.net,+1-219-560-0133,Robinsonshire,Fiji,25,Male,2024-07-29,2025-06-21,false,Silver,3348.16,Angel Hill,659,High,2026-05-19T05:33:22.514Z
6fc4253a-e2de-41ac-9269-01fb03bc4b22,Adam,Shaffer,jpeterson@example.org,001-851-316-1559x40781,Mariastad,Morocco,61,Non-binary,2025-06-04,2025-07-26,true,Platinum,476.74,Adam Shaffer,349,Low,2026-05-19T05:33:22.514Z
e0dd6a1a-bcfa-4206-a205-ac0efbf85c97,Chad,Stanley,tracie31@example.com,575-425-5341x928,Grayside,Cyprus,31,Male,2022-05-04,2022-06-20,true,Silver,10740.29,Chad Stanley,1476,VIP,2026-05-19T05:33:22.514Z
791a4371-6212-4c56-906d-05326d831399,Megan,Parsons,lydiatrujillo@example.com,513-695-3767x242,Hurstfurt,Guyana,52,Female,2024-10-19,2025-10-13,true,Platinum,8838.99,Megan Parsons,577,VIP,2026-05-19T05:33:22.514Z
047dcf0d-6301-476d-9df8-2dec3fc517ba,Whitney,Hicks,camposmichelle@example.org,+1-326-691-6697x8480,Danielchester,Cayman Islands,66,Male,2022-06-22,2022-06-25,false,Gold,4168.07,Whitney Hicks,1427,High,2026-05-19T05:33:22.514Z
0cd66ff2-2c4f-4f91-9da5-ae9b38e7ef65,Stephanie,Liu,teresa28@example.org,+1-948-293-2528x8095,West Natashaport,Dominican Republic,66,Female,2023-08-24,2023-12-12,true,Bronze,5698.91,Stephanie Liu,999,High,2026-05-19T05:33:22.514Z
b0def78c-77c3-4e27-b5ee-d1f98dc183ee,Natalie,Arroyo,dennislisa@example.net,001-822-778-2489x63834,Lammouth,British Indian Ocean Territory (Chagos Archipelago),72,Female,2021-08-31,2022-03-02,false,Bronze,10945.98,Natalie Arroyo,1722,VIP,2026-05-19T05:33:22.514Z
2d4ec29b-958f-491f-a583-649e977b4af8,Caleb,Smith,briannasmith@example.net,+1-403-910-5183x4738,Port Thomas,Ireland,25,Female,2022-08-13,2023-05-14,true,Gold,12441.07,Caleb Smith,1375,VIP,2026-05-19T05:33:22.514Z
5a0f81be-23e9-4440-b877-4c3b5bdcfe5c,Collin,Jordan,ujenkins@example.org,+1-610-965-1333x87262,Port Sandra,Kenya,74,Female,2022-04-30,2023-03-12,true,Bronze,687.37,Collin Jordan,1480,Medium,2026-05-19T05:33:22.514Z
ba972d0d-57d8-4309-ad95-c6604126be00,Melissa,Garcia,novaksara@example.org,(267)873-6026,Saramouth,France,67,Female,2025-06-02,2025-09-26,true,Silver,12997.26,Melissa Garcia,351,VIP,2026-05-19T05:33:22.514Z
